# 04 — Workflow Machine Learning complet

**Objectif :** reproduire le workflow professionnel : données → split → pipeline → modèles → évaluation honnête.
Dataset : `make_classification` de scikit-learn (fonctionne **hors ligne**, aucun téléchargement).

In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             average_precision_score, precision_score, recall_score)

# Dataset déséquilibré (comme la fraude, la maladie, le défaut) : 15 % de positifs
X, y = make_classification(n_samples=4000, n_features=18, n_informative=8,
                           weights=[0.85, 0.15], flip_y=0.05, random_state=0)
print("Forme :", X.shape, "| taux de positifs :", y.mean().round(3))


In [ ]:
# 1) SÉPARER D'ABORD — toute transformation vient après (anti-fuite)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                          random_state=42, stratify=y)
print("Train :", X_tr.shape, "| Test :", X_te.shape)

# 2) BASELINE : prédire toujours la classe majoritaire
baseline = np.zeros(len(y_te), dtype=int)
print("Baseline — accuracy :", round((baseline == y_te).mean(), 3),
      "| rappel :", recall_score(y_te, baseline, zero_division=0))
# 👉 Conclusion : l'accuracy est trompeuse, le rappel aussi. On choisira donc ROC-AUC / PR-AUC.


In [ ]:
# 3) PIPELINE (imputation + normalisation dans le pipeline = pas de fuite)
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]),
     list(range(X.shape[1]))),
])

modeles = {
    "logistique": Pipeline([("prep", prep),
                            ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))]),
    "random_forest": Pipeline([("prep", prep),
                               ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                                              class_weight="balanced_subsample",
                                                              n_jobs=-1, random_state=42))]),
}

# 4) VALIDATION CROISÉE sur le train uniquement
for nom, modele in modeles.items():
    cv = cross_val_score(modele, X_tr, y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                         scoring="roc_auc", n_jobs=-1)
    print(f"{nom:14s} CV ROC-AUC = {cv.mean():.3f} (±{cv.std():.3f})")


In [ ]:
# 5) ENTRAÎNER puis évaluer sur le test (on le touche maintenant, une fois)
resultats = {}
for nom, modele in modeles.items():
    modele.fit(X_tr, y_tr)
    p = modele.predict_proba(X_te)[:, 1]
    resultats[nom] = p
    print(f"{nom:14s} AUC={roc_auc_score(y_te, p):.3f} PR-AUC={average_precision_score(y_te, p):.3f}")

# TODO : pour le meilleur modèle, trouve le seuil qui donne un rappel >= 0.70
# avec la meilleure précision possible. Indice : boucle sur np.arange(0.05, 0.95, 0.01)
meilleur = ...
seuil_optimal = ...
print("seuil retenu :", seuil_optimal)


In [ ]:
# 6) Rapport final + matrice de confusion
p = resultats[meilleur]
pred = (p >= seuil_optimal).astype(int)
print(confusion_matrix(y_te, pred))
print(classification_report(y_te, pred, digits=3))
# 👉 Interprète en 3 lignes : combien de cas détectés ? combien de fausses alertes ? est-ce acceptable ?

# 7) Écart train/test = détection de surapprentissage
m = modeles[meilleur].fit(X_tr, y_tr)
print("AUC train :", round(roc_auc_score(y_tr, m.predict_proba(X_tr)[:, 1]), 3),
      "| AUC test :", round(roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]), 3))


## 🛠️ Exercices
1. Ajoute une colonne **catégorielle** (`ville`) au dataset et intègre-la au pipeline avec `OneHotEncoder(handle_unknown="ignore")`.
2. Fais varier `max_depth` de la forêt (2, 5, 10, None) et note l'écart train/test. Explique le surapprentissage avec tes chiffres.
3. **Fuite volontaire** : ajoute à X une colonne parfaitement corrélée à y (`y + bruit léger`). Observe l'AUC passer à 0,99. Retire-la. **C'est exactement ce qu'il faut savoir détecter en entretien.**
4. Applique toute la démarche à **ton dataset fil rouge** et écris un README.

### ✅ Auto-évaluation
- [ ] Je sépare les données avant toute transformation
- [ ] Je choisis la métrique selon le coût des erreurs
- [ ] Je sais détecter un surapprentissage
- [ ] Je sais repérer une fuite de données
